# 🔀 Notebook 5: LangGraph Agent Workflow
## 9-Node State Machine with Self-Reflection Loop

This notebook walks through the **LangGraph agent orchestration** — the brain of the system:

```
START → memory_node → rewrite_node → router_node
                                         │
                    ┌────────────────────┼────────────────┐
                    ▼                    ▼                ▼
               rag_node            kg_node          tool_node
                    │                    │                │
                    └──── combined ──────┴────────────────┘
                                         │
                                    answer_node
                                         │
                                    reflect_node ◄── retry loop (max 2)
                                         │
                              fact_extraction_node → END
```

Key features:
- **Query rewriting** — resolves pronouns and implicit references
- **Intelligent routing** — LLM classifies each query into the optimal path
- **Self-reflection** — detects low-confidence answers and retries
- **Automatic fact extraction** — learns durable facts from the conversation

In [ ]:
import sys
sys.path.insert(0, '..')

from src.agent.state import ChatState
from src.agent.graph import build_graph

print('✅ Agent modules loaded!')

## Step 1: State Schema

The `ChatState` TypedDict defines all data flowing through the graph.

In [ ]:
# Inspect the state schema
print('📋 ChatState Fields:')
print(f'{"Field":<25} {"Type"}')
print('-' * 55)
for field, type_hint in ChatState.__annotations__.items():
    print(f'  {field:<23} {str(type_hint)}')

print(f'\n  Total fields: {len(ChatState.__annotations__)}')

# Highlight the unique fields
unique_fields = ['rewritten_query', 'confidence', 'reflection_count', 'needs_retry', 'latency', 'provider_used']
print(f'\n🌟 Unique fields (not in any competitor):')
for f in unique_fields:
    if f in ChatState.__annotations__:
        print(f'  ✅ {f}: {ChatState.__annotations__[f]}')

## Step 2: Graph Compilation

The LangGraph workflow compiles into an executable graph.

In [ ]:
# Build and inspect the graph
graph = build_graph()
print(f'✅ Graph compiled successfully!')
print(f'\n📊 Graph structure:')
if hasattr(graph, 'nodes'):
    print(f'   Nodes: {len(graph.nodes)}')
    for node_name in graph.nodes:
        print(f'     • {node_name}')

# Print the routing table
print(f'\n🚦 Routing Table:')
routes = {
    'rag': 'Knowledge base questions → vector retrieval + graph',
    'kg': 'Relationship questions → knowledge graph traversal',
    'tool': 'Dynamic queries → one of 12 tools (weather, stocks, etc.)',
    'direct': 'General/personal questions → answer from memory + LLM',
    'hybrid': 'Complex questions → both RAG and KG combined',
}
for route, desc in routes.items():
    print(f'   {route:<8} → {desc}')

## Step 3: Query Rewriting Demo

The `rewrite_node` resolves pronouns and implicit references using chat history context.

In [ ]:
# Demo: Query rewriting examples
rewrite_examples = [
    {
        'history': [{'role': 'user', 'content': 'Tell me about Bitcoin'}],
        'query': 'What about its price?',
        'expected': 'What is the current price of Bitcoin?',
    },
    {
        'history': [{'role': 'user', 'content': 'I prefer Python'}],
        'query': 'Suggest a project for me',
        'expected': 'Suggest a Python project that I would enjoy',
    },
    {
        'history': [],
        'query': 'What is deep learning?',
        'expected': 'What is deep learning?',  # No rewrite needed
    },
]

print('🔄 Query Rewriting Examples:')
print(f'{"Original":<35} {"Context":<30} {"Expected Rewrite"}')
print('-' * 100)
for ex in rewrite_examples:
    context = ex['history'][-1]['content'] if ex['history'] else '(none)'
    print(f'  {ex["query"]:<33} {context:<28} {ex["expected"]}')

print('\n💡 The rewriter uses the fast Llama 8B model on Groq for sub-50ms latency.')

## Step 4: Self-Reflection Loop

After generating an answer, the `reflect_node` checks:
1. Does the answer address the question? (semantic check)
2. Is confidence > 0.5? (self-assessed)
3. Did the answer use the retrieved context? (grounding check)

If any check fails AND `reflection_count < 2`: **retry with expanded retrieval**.

In [ ]:
from src.agent.graph import reflect_node

# Test cases for self-reflection
test_cases = [
    {
        'answer': 'Machine learning is a subset of AI that enables systems to learn from data and improve.',
        'confidence': 0.92,
        'reflection_count': 0,
        'message': 'What is machine learning?',
        'expected': 'ACCEPT (high confidence)',
    },
    {
        'answer': 'I am not sure about that.',
        'confidence': 0.2,
        'reflection_count': 0,
        'message': 'What is quantum computing?',
        'expected': 'RETRY (low confidence, 0 retries)',
    },
    {
        'answer': 'I am still not confident.',
        'confidence': 0.3,
        'reflection_count': 2,
        'message': 'Explain string theory',
        'expected': 'ACCEPT (max retries reached)',
    },
]

print('🔄 Self-Reflection Test Cases:')
for tc in test_cases:
    result = reflect_node(tc)
    action = 'RETRY' if result.get('needs_retry', False) else 'ACCEPT'
    print(f'\n  Question: "{tc["message"]}"')
    print(f'  Confidence: {tc["confidence"]} | Retries: {tc["reflection_count"]}')
    print(f'  Decision: {action} (expected: {tc["expected"]})')

## Step 5: End-to-End Agent Invocation

> **Note:** This requires an LLM provider configured in `.env`.

In [ ]:
# Full agent invocation (requires LLM)
try:
    result = graph.invoke({
        'user_id': 'notebook_user',
        'message': 'What is retrieval augmented generation?',
    })
    
    print('🤖 Agent Response:')
    print(f'   Route: {result.get("route", "unknown")}')
    print(f'   Confidence: {result.get("confidence", "N/A")}')
    print(f'   Provider: {result.get("provider_used", "N/A")}')
    print(f'   Answer: {result.get("answer", "N/A")[:300]}...')
    if result.get('latency'):
        print(f'   Latency: {result["latency"]}')
except Exception as e:
    print(f'⚠️ Full agent invocation requires an LLM provider.')
    print(f'   Configure GROQ_API_KEY or GOOGLE_API_KEY in .env')
    print(f'   Error: {e}')